In [1]:
!uv pip install -q "sagemaker>=2.0.0,<3.0.0" "sagemaker[local]" "boto3==1.42.82" "pytz==2026.1.post1" "requests==2.33.1"

In [2]:
!uv pip freeze | grep -E "sagemaker|boto3|pytz|requests"

boto3==1.42.82
pytz==2026.1.post1
requests==2.33.1
sagemaker==2.257.1
sagemaker-core==1.0.77


In [3]:
import os
import subprocess
import sys

import boto3
import requests
from requests.auth import HTTPBasicAuth

In [4]:
!docker build --no-cache -q -t credit-risk-training:latest -f /app/give_me_some_credit/sagemaker/training/Dockerfile /app/give_me_some_credit/sagemaker/training/

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            Install the buildx component to build images with BuildKit:
            https://docs.docker.com/go/buildx/

sha256:a55356abf7216e2a6071cee78933b05d48adc7cc5570576f4e7fa0e5a945095e


In [5]:
!aws s3 ls s3://data-lake --recursive

2026-03-29 19:25:29        402 BaselineTraining-1774812261-97ad/output/model.tar.gz
2026-03-29 19:25:29        184 BaselineTraining-1774812261-97ad/output/output.tar.gz
2026-03-29 19:27:03        837 HyperparameterTuning-1774812329-50a2/output/model.tar.gz
2026-03-29 19:27:03        184 HyperparameterTuning-1774812329-50a2/output/output.tar.gz
2026-03-29 19:04:33          0 bronze/
2026-03-29 19:17:35    7564965 bronze/credit_risk/kaggle/2026-03-29/cs-training.csv
2026-03-29 19:19:38      11332 credit-risk-training-2026-03-29-19-19-38-097/input/code/preprocess.py
2026-03-29 19:19:38      10840 credit-risk-training-2026-03-29-19-19-38-105/input/code/evaluate.py
2026-03-29 19:19:38      11332 credit-risk-training-2026-03-29-19-19-38-111/input/code/preprocess.py
2026-03-29 19:19:38      10840 credit-risk-training-2026-03-29-19-19-38-117/input/code/evaluate.py
2026-03-29 19:23:58      11332 credit-risk-training-2026-03-29-19-23-57-994/input/code/preprocess.py
2026-03-29 19:23:58      10840

In [6]:
def get_latest_ingestion_date(bucket, prefix, s3_endpoint=None):
    s3 = boto3.client("s3", endpoint_url=s3_endpoint)

    # We use delimiter='/' to get "folders" inside CommonPrefixes
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix, Delimiter="/")

    dates = []
    for page in pages:
        for obj in page.get("CommonPrefixes", []):
            # Extract 'YYYY-MM-DD' from 'path/ingestion_date=YYYY-MM-DD/'
            folder_name = obj.get("Prefix").split("=")[-1].strip("/")
            dates.append(folder_name)

    if not dates:
        raise ValueError(f"No partitions found in s3://{bucket}/{prefix}")

    # Sorting ensures '2026-03-22' comes after '2026-03-21'
    return sorted(dates)[-1]


# --- Usage in your script ---
S3_ENDPOINT = "http://localstack:4566"
LATEST_DATE = get_latest_ingestion_date(
    bucket="data-lake", prefix="gold/credit_risk/features/", s3_endpoint=S3_ENDPOINT
)

print(f"Detected Latest Date: {LATEST_DATE}")

Detected Latest Date: 2026-03-29


In [7]:
NETWORK = os.environ["NETWORK"]
result = subprocess.run(
    [
        sys.executable,
        "/app/give_me_some_credit/sagemaker/training/sm_pipeline.py",
        "--mode",
        "local",
        "--ingestion-date",
        LATEST_DATE,
        "--s3-endpoint",
        "http://localstack:4566",
        "--n-trials",
        "3",
        "--auc-threshold",
        "0.85",
        "--network",
        NETWORK,
    ],
    capture_output=False,
    text=True,
)
print(f"\nExit code: {result.returncode}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


INFO:sagemaker_pipeline_give_me_some_credit:Args: {'mode': 'local', 'ingestion_date': '2026-03-29', 's3_endpoint': 'http://localstack:4566', 's3_bucket': 'data-lake', 'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'credit_risk_pipeline', 'n_trials': 3, 'auc_threshold': 0.85, 'training_image': 'credit-risk-training:latest', 'network': 'mlops-lab_mlops-lab-net', 'aws_region': 'us-east-1'}
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:botocore.configprovider:Found endpoint for s3 via: environment_global.
INFO:sagemaker_session:Local SageMaker session ready (bucket=data-lake)
INFO:botocore.credentials:Found credentials in environment variables.
INFO:botocore.configprovider:Found endpoint for sagemaker via: environment_global.
INFO:botocore.configprovider:Found endpoint for sagemaker via: environment_global.
INFO:botocore.configprovider:Found endpoint for scheduler via: environment_global.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python

time="2026-03-29T19:33:17Z" level=warning msg="/tmp/tmp81o8mv3v/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-03-29T19:33:17Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmp81o8mv3v\".\nSet `external: true` to use an existing network"
 Container a6fto12jrd-algo-1-ccsat Creating 
 Container a6fto12jrd-algo-1-ccsat Created 
Attaching to a6fto12jrd-algo-1-ccsat
 Container a6fto12jrd-algo-1-ccsat Starting 
 Container a6fto12jrd-algo-1-ccsat Started 
a6fto12jrd-algo-1-ccsat  | 2026-03-29 19:33:21,816 - preprocess - INFO - Args: {'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'credit_risk_pipeline', 'random_state': 42}
a6fto12jrd-algo-1-ccsat  | 2026-03-29 19:33:21,955 - preprocess - INFO - Loaded 149,390 rows, 18 columns
a6fto12jrd-algo-1-ccsat  | 2026-03-29 19:33:22,014 - preprocess - INFO -   train: 104,573 rows | default rate: 6.70%
a6fto1

INFO:sagemaker.local.image:===== Job Complete =====
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.
INFO:sagemaker.local.entities:Pipeline step 'BaselineTraining' SUCCEEDED.
INFO:sagemaker.local.entities:Starting pipeline step: 'HyperparameterTuning'
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' found using Docker Compose CLI.
INFO:sagemaker.local.local_session:Starting training job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:


jl9dyls2fk-algo-1-574f3  | 2026-03-29 19:34:08,570 - train_step - INFO - [CV-5] AUC=0.8632 ± 0.0056
jl9dyls2fk-algo-1-574f3  | 2026-03-29 19:34:11,819 - train_step - INFO - [train] AUC=0.8789 KS=0.5999
jl9dyls2fk-algo-1-574f3  | 2026-03-29 19:34:11,844 - train_step - INFO - [val] AUC=0.8698 KS=0.5802
jl9dyls2fk-algo-1-574f3  | 2026/03/29 19:34:14 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
jl9dyls2fk-algo-1-574f3  | 🏃 View run baseline_catboost at: http://mlflow:5000/#/experiments/1/runs/07b4f2566b6e4049899977c9fa63161d
jl9dyls2fk-algo-1-574f3  | 🧪 View experiment at: http://mlflow:5000/#/experiments/1
jl9dyls2fk-algo-1-574f3  | 2026-03-29 19:34:14,126 - train_step - INFO - Baseline summary:
jl9dyls2fk-algo-1-574f3  | 2026-03-29 19:34:14,126 - train_step - INFO -   catboost: val_auc=0.8698 ks=0.5802
jl9dyls2fk-algo-1-574f3  | 2026-03-29 19:34:14,126 - trai

INFO:sagemaker.local.image:===== Job Complete =====
INFO:botocore.configprovider:Found endpoint for sts via: environment_global.
INFO:sagemaker.local.entities:Pipeline step 'HyperparameterTuning' SUCCEEDED.
INFO:sagemaker.local.entities:Starting pipeline step: 'Evaluation'
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.local.image:'Docker Compose' found using Docker Compose CLI.
INFO:sagemaker.local.local_session:Starting processing job
INFO:sagemaker.local.image:Using the long-lived AWS credentials found in session
INFO:sagemaker.local.image:docker compose file: 
networks:
  sagemaker-local:
    

ssn88dqjnj-algo-1-ft8xt  | 2026-03-29 19:36:25,716 - tune_step - INFO - Tuning complete.
ssn88dqjnj-algo-1-ft8xt exited with code 0
 Compose Stopping Aborting on container exit...
 Container ssn88dqjnj-algo-1-ft8xt Stopping 
 Container ssn88dqjnj-algo-1-ft8xt Stopped 
time="2026-03-29T19:36:27Z" level=warning msg="/tmp/tmpqcnb93o5/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-03-29T19:36:27Z" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmpqcnb93o5\".\nSet `external: true` to use an existing network"
 Container ji5eoyrh03-algo-1-8j24m Creating 
 Container ji5eoyrh03-algo-1-8j24m Created 
Attaching to ji5eoyrh03-algo-1-8j24m
 Container ji5eoyrh03-algo-1-8j24m Starting 
 Container ji5eoyrh03-algo-1-8j24m Started 
ji5eoyrh03-algo-1-8j24m  | 2026-03-29 19:36:32,336 - evaluate - INFO - Args: {'mlflow_uri': 'http://mlflow:5000', 'experiment_name': 'cred